# 🚴 Citi Bike Trip Data — Big Data & NoSQL Project

Dataset: NYC Citi Bike trips — March 2019 to December 2019  
Covers: Data cleaning, feature engineering, analytical queries, SparkML, bonus dashboard queries, and Power BI export.
Dashboard:https://alialnaggar.github.io/CitiBike_Bigdata_Project/$0
---

## ⚙️ Phase 0 — Environment Setup

In [ ]:
!pip install pyspark gdown --quiet

In [ ]:
import gdown

file_id     = '1eHda5_Rs9eohbVyl5FjYs_JlAKGb6ik-'
output_path = 'citibike.csv'

gdown.download(f'https://drive.google.com/uc?id={file_id}', output_path, quiet=False)
print('✅ Dataset downloaded.')

Downloading...
From (original): https://drive.google.com/uc?id=1eHda5_Rs9eohbVyl5FjYs_JlAKGb6ik-
From (redirected): https://drive.google.com/uc?id=1eHda5_Rs9eohbVyl5FjYs_JlAKGb6ik-&confirm=t&uuid=1816cb27-ecd8-4f4a-8df3-21fb4e377853
To: /content/citibike.csv
100%|██████████| 241M/241M [00:05<00:00, 41.0MB/s]


✅ Dataset downloaded.


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('CitiBike_Analysis') \
    .config('spark.sql.legacy.timeParserPolicy', 'LEGACY') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print('✅ Spark session initialized.')
print(f'Spark version: {spark.version}')

✅ Spark session initialized.
Spark version: 4.0.2


---
## 🧹 Phase 1 — Data Cleaning & Feature Engineering

### 1a. Load the Dataset

In [ ]:
df = spark.read.csv('citibike.csv', header=True, inferSchema=True)

print(f'✅ Data loaded.')
print(f'   Rows    : {df.count():,}')
print(f'   Columns : {len(df.columns)}')
print('\nRaw column names:')
print(df.columns)
print('\nSchema:')
df.printSchema()

✅ Data loaded.
   Rows    : 1,300,000
   Columns : 15

Raw column names:
['_c0', 'starttime', 'stoptime', 'start station id', 'start station name', 'start station latitude', 'start station longitude', 'end station id', 'end station name', 'end station latitude', 'end station longitude', 'bikeid', 'usertype', 'birth year', 'gender']

Schema:
root
 |-- _c0: integer (nullable = true)
 |-- starttime: string (nullable = true)
 |-- stoptime: string (nullable = true)
 |-- start station id: double (nullable = true)
 |-- start station name: string (nullable = true)
 |-- start station latitude: double (nullable = true)
 |-- start station longitude: double (nullable = true)
 |-- end station id: double (nullable = true)
 |-- end station name: string (nullable = true)
 |-- end station latitude: double (nullable = true)
 |-- end station longitude: double (nullable = true)
 |-- bikeid: integer (nullable = true)
 |-- usertype: string (nullable = true)
 |-- birth year: integer (nullable = true)
 |-- ge

In [ ]:
# Rename all columns — replace spaces with underscores and label the index column
rename_map = {
    '_c0'                     : 'ride_id',
    'starttime'               : 'starttime',
    'stoptime'                : 'stoptime',
    'start station id'        : 'start_station_id',
    'start station name'      : 'start_station_name',
    'start station latitude'  : 'start_station_lat',
    'start station longitude' : 'start_station_lon',
    'end station id'          : 'end_station_id',
    'end station name'        : 'end_station_name',
    'end station latitude'    : 'end_station_lat',
    'end station longitude'   : 'end_station_lon',
    'bikeid'                  : 'bike_id',
    'usertype'                : 'user_type',
    'birth year'              : 'birth_year',
    'gender'                  : 'gender'
}

for old_name, new_name in rename_map.items():
    if old_name in df.columns:
        df = df.withColumnRenamed(old_name, new_name)

print('✅ Columns renamed.')
print(df.columns)

✅ Columns renamed.
['ride_id', 'starttime', 'stoptime', 'start_station_id', 'start_station_name', 'start_station_lat', 'start_station_lon', 'end_station_id', 'end_station_name', 'end_station_lat', 'end_station_lon', 'bike_id', 'user_type', 'birth_year', 'gender']


### 1b. Data Exploration & Cleaning

In [ ]:
df.show(5, truncate=False)

+-------+-----------------------+-----------------------+----------------+------------------------+-----------------+-----------------+--------------+-----------------------------+---------------+---------------+-------+----------+----------+------+
|ride_id|starttime              |stoptime               |start_station_id|start_station_name      |start_station_lat|start_station_lon|end_station_id|end_station_name             |end_station_lat|end_station_lon|bike_id|user_type |birth_year|gender|
+-------+-----------------------+-----------------------+----------------+------------------------+-----------------+-----------------+--------------+-----------------------------+---------------+---------------+-------+----------+----------+------+
|0      |2019-04-17 14:37:03.844|2019-04-17 14:43:13.767|264.0           |Maiden Ln & Pearl St    |40.70706456      |-74.00731853     |330.0         |Reade St & Broadway          |40.71450451    |-74.00562789   |16906  |Subscriber|1969      |1     |


In [ ]:
df.describe().show(truncate=False)

+-------+-----------------+-----------------------+-----------------------+------------------+---------------------+--------------------+--------------------+------------------+---------------------+--------------------+--------------------+------------------+----------+------------------+------------------+
|summary|ride_id          |starttime              |stoptime               |start_station_id  |start_station_name   |start_station_lat   |start_station_lon   |end_station_id    |end_station_name     |end_station_lat     |end_station_lon     |bike_id           |user_type |birth_year        |gender            |
+-------+-----------------+-----------------------+-----------------------+------------------+---------------------+--------------------+--------------------+------------------+---------------------+--------------------+--------------------+------------------+----------+------------------+------------------+
|count  |1300000          |1300000                |1300000            

In [ ]:
from pyspark.sql import functions as F

print('=== NULL Count per Column ===')
df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns
]).show(truncate=False)

=== NULL Count per Column ===
+-------+---------+--------+----------------+------------------+-----------------+-----------------+--------------+----------------+---------------+---------------+-------+---------+----------+------+
|ride_id|starttime|stoptime|start_station_id|start_station_name|start_station_lat|start_station_lon|end_station_id|end_station_name|end_station_lat|end_station_lon|bike_id|user_type|birth_year|gender|
+-------+---------+--------+----------------+------------------+-----------------+-----------------+--------------+----------------+---------------+---------------+-------+---------+----------+------+
|0      |0        |0       |33              |33                |0                |0                |33            |33              |0              |0              |0      |0        |0         |0     |
+-------+---------+--------+----------------+------------------+-----------------+-----------------+--------------+----------------+---------------+---------------+--

In [ ]:
string_cols = [f.name for f in df.schema.fields if str(f.dataType) == 'StringType()']

print('=== Empty String Count per String Column ===')
df.select([
    F.count(F.when(F.col(c) == '', c)).alias(c) for c in string_cols
]).show(truncate=False)

=== Empty String Count per String Column ===
+---------+--------+------------------+----------------+---------+
|starttime|stoptime|start_station_name|end_station_name|user_type|
+---------+--------+------------------+----------------+---------+
|0        |0       |0                 |0               |0        |
+---------+--------+------------------+----------------+---------+



In [ ]:
# Replace empty strings with NULL in string columns
for col_name in string_cols:
    df = df.withColumn(col_name,
        F.when(F.col(col_name) == '', None).otherwise(F.col(col_name))
    )

# Replace 0.0 in coordinate columns with NULL (not a valid NYC GPS location)
for col_name in ['start_station_lat', 'start_station_lon',
                 'end_station_lat',   'end_station_lon']:
    df = df.withColumn(col_name,
        F.when(F.col(col_name) == 0.0, None).otherwise(F.col(col_name))
    )

# Replace birth_year = 0 with NULL
df = df.withColumn('birth_year',
    F.when(F.col('birth_year') == 0, None).otherwise(F.col('birth_year'))
)

print('✅ All implicit nulls standardized.')
print(f'Total rows: {df.count():,}')

✅ All implicit nulls standardized.
Total rows: 1,300,000


In [ ]:
# Cast timestamps using SQL CAST which handles the milliseconds natively.
# Also cast station IDs from double (264.0) to int for clean comparisons.

df.createOrReplaceTempView('df_raw')

df = spark.sql("""
    SELECT
        ride_id,
        CAST(starttime AS TIMESTAMP) AS starttime,
        CAST(stoptime  AS TIMESTAMP) AS stoptime,
        CAST(start_station_id AS INT) AS start_station_id,
        start_station_name,
        start_station_lat,
        start_station_lon,
        CAST(end_station_id AS INT) AS end_station_id,
        end_station_name,
        end_station_lat,
        end_station_lon,
        bike_id,
        user_type,
        birth_year,
        gender
    FROM df_raw
""")

print('✅ Timestamps and station IDs cast correctly.')
df.select('starttime', 'stoptime', 'start_station_id', 'end_station_id').show(5, truncate=False)

null_ts = df.filter(F.col('starttime').isNull() | F.col('stoptime').isNull()).count()
print(f'Rows with NULL timestamps after cast: {null_ts}')

✅ Timestamps and station IDs cast correctly.
+-----------------------+-----------------------+----------------+--------------+
|starttime              |stoptime               |start_station_id|end_station_id|
+-----------------------+-----------------------+----------------+--------------+
|2019-04-17 14:37:03.844|2019-04-17 14:43:13.767|264             |330           |
|2019-04-17 14:37:01.225|2019-04-17 14:42:48.108|411             |438           |
|2019-04-17 14:37:06.936|2019-04-17 14:52:25.604|128             |358           |
|2019-04-17 14:37:02.985|2019-04-17 14:46:53.331|349             |3435          |
|2019-04-17 14:37:03.8  |2019-04-17 14:42:34.71 |3556            |3563          |
+-----------------------+-----------------------+----------------+--------------+
only showing top 5 rows
Rows with NULL timestamps after cast: 0


### 1c. Feature Engineering

In [ ]:
# ── Rider Age ─────────────────────────────────────────────────────────────────
# Age = trip year - birth year. Using F.year('starttime') keeps it accurate
# across the full dataset span (March-December 2019).

df = df.withColumn('Age',
    F.when(
        F.col('birth_year').isNotNull(),
        F.year('starttime') - F.col('birth_year').cast('int')
    ).otherwise(None)
)

print('✅ Age column created.')
df.select('birth_year', F.year('starttime').alias('trip_year'), 'Age').show(5)

✅ Age column created.
+----------+---------+---+
|birth_year|trip_year|Age|
+----------+---------+---+
|      1969|     2019| 50|
|      1974|     2019| 45|
|      1969|     2019| 50|
|      1986|     2019| 33|
|      1979|     2019| 40|
+----------+---------+---+
only showing top 5 rows


In [ ]:
# ── Trip Duration (seconds) ───────────────────────────────────────────────────
# unix_timestamp converts Timestamp to seconds since epoch.
# Subtracting start from stop gives elapsed ride time in seconds.

df = df.withColumn('Trip_Duration_Sec',
    (F.unix_timestamp('stoptime') - F.unix_timestamp('starttime')).cast('int')
)

print('✅ Trip_Duration_Sec column created.')
df.select('starttime', 'stoptime', 'Trip_Duration_Sec').show(5)

✅ Trip_Duration_Sec column created.
+--------------------+--------------------+-----------------+
|           starttime|            stoptime|Trip_Duration_Sec|
+--------------------+--------------------+-----------------+
|2019-04-17 14:37:...|2019-04-17 14:43:...|              370|
|2019-04-17 14:37:...|2019-04-17 14:42:...|              347|
|2019-04-17 14:37:...|2019-04-17 14:52:...|              919|
|2019-04-17 14:37:...|2019-04-17 14:46:...|              591|
|2019-04-17 14:37:...|2019-04-17 14:42:...|              331|
+--------------------+--------------------+-----------------+
only showing top 5 rows


In [ ]:
# ── Trip Distance via Haversine Formula (UDF) ─────────────────────────────────
# Haversine formula calculates shortest distance over Earth surface
# between two GPS coordinates. Output in kilometers.

import math
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType, StringType

def haversine(lat1, lon1, lat2, lon2):
    """Returns great-circle distance in km between two GPS points."""
    if None in (lat1, lon1, lat2, lon2):
        return None
    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat / 2) ** 2 +
         math.cos(math.radians(lat1)) *
         math.cos(math.radians(lat2)) *
         math.sin(dlon / 2) ** 2)
    return round(R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a)), 4)

haversine_udf = udf(haversine, DoubleType())

df = df.withColumn('Trip_Distance_Km',
    haversine_udf(
        F.col('start_station_lat'), F.col('start_station_lon'),
        F.col('end_station_lat'),   F.col('end_station_lon')
    )
)

print('✅ Trip_Distance_Km created using Haversine UDF.')
df.select('start_station_lat','start_station_lon',
          'end_station_lat','end_station_lon','Trip_Distance_Km').show(5)

✅ Trip_Distance_Km created using Haversine UDF.
+-----------------+-----------------+---------------+---------------+----------------+
|start_station_lat|start_station_lon|end_station_lat|end_station_lon|Trip_Distance_Km|
+-----------------+-----------------+---------------+---------------+----------------+
|      40.70706456|     -74.00731853|    40.71450451|   -74.00562789|          0.8395|
|      40.72228087|     -73.97668709|    40.72779126|   -73.98564945|          0.9725|
|      40.72710258|     -74.00297088|    40.73291553|   -74.00711384|          0.7346|
|      40.71850211|     -73.98329859|      40.718822|      -73.99596|          1.0677|
|       40.7527085|      -73.9397405|      40.757186|     -73.932719|          0.7731|
+-----------------+-----------------+---------------+---------------+----------------+
only showing top 5 rows


In [ ]:
# ── Trip Speed (km/h) via UDF ─────────────────────────────────────────────────
# Speed = Distance / Time. Duration converted from seconds to hours.

def calc_speed_kmh(distance_km, duration_sec):
    """Returns riding speed in km/h. Returns None for invalid inputs."""
    if distance_km is None or duration_sec is None or duration_sec <= 0:
        return None
    return round(distance_km / (duration_sec / 3600.0), 4)

speed_udf = udf(calc_speed_kmh, DoubleType())

df = df.withColumn('Trip_Speed_Kmh',
    speed_udf(F.col('Trip_Distance_Km'), F.col('Trip_Duration_Sec'))
)

print('✅ Trip_Speed_Kmh created using UDF.')
df.select('Trip_Distance_Km', 'Trip_Duration_Sec', 'Trip_Speed_Kmh').show(5)

✅ Trip_Speed_Kmh created using UDF.
+----------------+-----------------+--------------+
|Trip_Distance_Km|Trip_Duration_Sec|Trip_Speed_Kmh|
+----------------+-----------------+--------------+
|          0.8395|              370|        8.1681|
|          0.9725|              347|       10.0893|
|          0.7346|              919|        2.8776|
|          1.0677|              591|        6.5038|
|          0.7731|              331|        8.4083|
+----------------+-----------------+--------------+
only showing top 5 rows


In [ ]:
# ── Period of Day ─────────────────────────────────────────────────────────────
# Morning 06-11 | Afternoon 12-17 | Evening 18-21 | Night 22-05

df = df.withColumn('Start_Hour', F.hour('starttime'))

df = df.withColumn('Period_of_Day',
    F.when((F.col('Start_Hour') >= 6)  & (F.col('Start_Hour') < 12), 'Morning')
     .when((F.col('Start_Hour') >= 12) & (F.col('Start_Hour') < 18), 'Afternoon')
     .when((F.col('Start_Hour') >= 18) & (F.col('Start_Hour') < 22), 'Evening')
     .otherwise('Night')
)

print('✅ Period_of_Day column created.')
df.groupBy('Period_of_Day').count().orderBy('Period_of_Day').show()

✅ Period_of_Day column created.
+-------------+------+
|Period_of_Day| count|
+-------------+------+
|    Afternoon|564756|
|      Evening|238265|
|      Morning|423038|
|        Night| 73941|
+-------------+------+



In [ ]:
# ── Start Month ───────────────────────────────────────────────────────────────
df = df.withColumn('Start_Month', F.month('starttime'))

print('✅ Start_Month column created.')
df.select('starttime', 'Start_Month').show(5)

✅ Start_Month column created.
+--------------------+-----------+
|           starttime|Start_Month|
+--------------------+-----------+
|2019-04-17 14:37:...|          4|
|2019-04-17 14:37:...|          4|
|2019-04-17 14:37:...|          4|
|2019-04-17 14:37:...|          4|
|2019-04-17 14:37:...|          4|
+--------------------+-----------+
only showing top 5 rows


In [ ]:
# ── Validation after Feature Engineering ─────────────────────────────────────
new_cols = ['Age', 'Trip_Duration_Sec', 'Trip_Distance_Km',
            'Trip_Speed_Kmh', 'Period_of_Day', 'Start_Month', 'Start_Hour']

print('=== Schema of Engineered Columns ===')
df.select(new_cols).printSchema()

print('=== Statistical Summary ===')
df.select('Age','Trip_Duration_Sec','Trip_Distance_Km','Trip_Speed_Kmh').describe().show()

print('=== Sample Rows ===')
df.select(new_cols).show(10, truncate=False)

=== Schema of Engineered Columns ===
root
 |-- Age: integer (nullable = true)
 |-- Trip_Duration_Sec: integer (nullable = true)
 |-- Trip_Distance_Km: double (nullable = true)
 |-- Trip_Speed_Kmh: double (nullable = true)
 |-- Period_of_Day: string (nullable = false)
 |-- Start_Month: integer (nullable = true)
 |-- Start_Hour: integer (nullable = true)

=== Statistical Summary ===
+-------+------------------+-----------------+------------------+-----------------+
|summary|               Age|Trip_Duration_Sec|  Trip_Distance_Km|   Trip_Speed_Kmh|
+-------+------------------+-----------------+------------------+-----------------+
|  count|           1300000|          1300000|           1300000|          1300000|
|   mean| 38.99095230769231|960.2060076923077|1.7747299653846262|8.768071153923307|
| stddev|12.169923825312448|9518.983077401252|1.4102711603485913| 3.36576543311731|
|    min|                16|               61|               0.0|              0.0|
|    max|               145|

### 1d. Noise Flagging
We flag noisy records with a boolean column — preserving raw data while clearly marking exclusions.

In [ ]:
df = df.withColumn('Is_Noisy',
    # Under 60 seconds — almost certainly a docking error
    (F.col('Trip_Duration_Sec') < 60) |
    # Over 40 km/h — physically impossible on a city bicycle
    (F.col('Trip_Speed_Kmh') > 40) |
    # Age out of realistic riding range
    (F.col('Age') > 100) |
    (F.col('Age') < 12)  |
    # Missing essential identifiers
    F.col('start_station_name').isNull() |
    F.col('end_station_name').isNull()   |
    F.col('bike_id').isNull()
)

noisy = df.filter(F.col('Is_Noisy') == True).count()
clean = df.filter(F.col('Is_Noisy') == False).count()

print(f'✅ Noise flagging complete.')
print(f'   Noisy records : {noisy:,}')
print(f'   Clean records : {clean:,}')
print(f'   Total records : {df.count():,}')

✅ Noise flagging complete.
   Noisy records : 688
   Clean records : 1,299,312
   Total records : 1,300,000


In [ ]:
# Cache the clean DataFrame — Spark reuses this instead of re-reading disk
df_clean = df.filter(F.col('Is_Noisy') == False)
df_clean.cache()
print(f'✅ df_clean cached. Rows: {df_clean.count():,}')

✅ df_clean cached. Rows: 1,299,312


In [ ]:
df_clean.createOrReplaceTempView('citibike')
print('✅ SQL view "citibike" registered.')

✅ SQL view "citibike" registered.


---
## 📊 Phase 2 — Analytical Queries

### Query (a) — Round Trip Percentage per User Type

In [ ]:
# A round trip: rider starts and ends at the same station.

round_trip_df = spark.sql("""
    SELECT
        user_type,
        COUNT(*) AS total_trips,
        SUM(CASE WHEN start_station_id = end_station_id THEN 1 ELSE 0 END) AS round_trips,
        ROUND(
            SUM(CASE WHEN start_station_id = end_station_id THEN 1 ELSE 0 END)
            * 100.0 / COUNT(*), 2
        ) AS round_trip_pct
    FROM citibike
    WHERE user_type IS NOT NULL
    GROUP BY user_type
    ORDER BY user_type
""")

round_trip_df.show(truncate=False)

print("""
📌 Business Interpretation:
Round trips signal recreational use. Customers (casual users) are expected to have a
higher round-trip rate since they ride for leisure. Subscribers (annual members) show
a lower rate as they commute point-to-point. This guides station placement strategy.
""")

+----------+-----------+-----------+--------------+
|user_type |total_trips|round_trips|round_trip_pct|
+----------+-----------+-----------+--------------+
|Customer  |183750     |9737       |5.30          |
|Subscriber|1115562    |17960      |1.61          |
+----------+-----------+-----------+--------------+


📌 Business Interpretation:
Round trips signal recreational use. Customers (casual users) are expected to have a
higher round-trip rate since they ride for leisure. Subscribers (annual members) show
a lower rate as they commute point-to-point. This guides station placement strategy.



### Query (b) — Most Popular Start Stations

In [ ]:
# Rank every start station by trip count using the RANK() window function.

popular_stations_df = spark.sql("""
    SELECT
        start_station_name,
        COUNT(*) AS trip_count,
        RANK() OVER (ORDER BY COUNT(*) DESC) AS usage_rank
    FROM citibike
    WHERE start_station_name IS NOT NULL
    GROUP BY start_station_name
    ORDER BY trip_count DESC
    LIMIT 20
""")

popular_stations_df.show(20, truncate=False)

print("""
📌 Business Interpretation:
Top-ranked start stations are the busiest origin points. City planners should prioritize
dock expansion and guaranteed bike availability here during morning peak hours.
""")

+-----------------------------+----------+----------+
|start_station_name           |trip_count|usage_rank|
+-----------------------------+----------+----------+
|Pershing Square North        |9761      |1         |
|8 Ave & W 31 St              |8001      |2         |
|E 17 St & Broadway           |7524      |3         |
|Broadway & E 22 St           |7041      |4         |
|W 21 St & 6 Ave              |7011      |5         |
|Broadway & E 14 St           |6973      |6         |
|West St & Chambers St        |6731      |7         |
|Broadway & W 60 St           |6596      |8         |
|Christopher St & Greenwich St|6557      |9         |
|12 Ave & W 40 St             |6446      |10        |
|W 20 St & 11 Ave             |6105      |11        |
|Lafayette St & E 8 St        |5810      |12        |
|W 41 St & 8 Ave              |5787      |13        |
|8 Ave & W 33 St              |5758      |14        |
|Broadway & W 25 St           |5612      |15        |
|W 31 St & 7 Ave            

### Query (c) — Rush Hours (Highest Demand by Hour)

In [ ]:
rush_hours_df = spark.sql("""
    SELECT
        Start_Hour AS hour_of_day,
        COUNT(*) AS trip_count
    FROM citibike
    GROUP BY Start_Hour
    ORDER BY trip_count DESC
""")

rush_hours_df.show(24, truncate=False)

print("""
📌 Business Interpretation:
Peak demand clusters in morning commute (7-9 AM) and evening commute (5-7 PM).
Off-peak hours are optimal maintenance windows.
""")

+-----------+----------+
|hour_of_day|trip_count|
+-----------+----------+
|17         |126578    |
|8          |115714    |
|18         |104873    |
|16         |98266     |
|15         |90935     |
|14         |88521     |
|13         |83816     |
|9          |79516     |
|12         |76324     |
|7          |70091     |
|11         |67774     |
|19         |64084     |
|10         |56660     |
|20         |41246     |
|6          |33068     |
|21         |27942     |
|22         |19573     |
|0          |12647     |
|23         |11566     |
|5          |10495     |
|1          |7939      |
|2          |5015      |
|4          |3573      |
|3          |3096      |
+-----------+----------+


📌 Business Interpretation:
Peak demand clusters in morning commute (7-9 AM) and evening commute (5-7 PM).
Off-peak hours are optimal maintenance windows.



### Query (d) — Age Group UDF & Trip Duration by Age Group

In [ ]:
# UDF classifies riders into: Young (12-25), Adult (26-60), Senior (61+)

def classify_age_group(age):
    if age is None: return 'Unknown'
    if 12 <= age <= 25: return 'Young'
    elif 26 <= age <= 60: return 'Adult'
    else: return 'Senior'

age_group_udf = udf(classify_age_group, StringType())

df_clean = df_clean.withColumn('Age_Group', age_group_udf(F.col('Age')))
df_clean.createOrReplaceTempView('citibike')

print('✅ Age_Group column added.')
df_clean.groupBy('Age_Group').count().orderBy('Age_Group').show()

✅ Age_Group column added.
+---------+-------+
|Age_Group|  count|
+---------+-------+
|    Adult|1089222|
|   Senior|  62929|
|    Young| 147161|
+---------+-------+



In [ ]:
age_duration_df = spark.sql("""
    SELECT
        Age_Group,
        COUNT(*) AS trip_count,
        ROUND(AVG(Trip_Duration_Sec), 2) AS avg_duration_sec,
        ROUND(MIN(Trip_Duration_Sec), 2) AS min_duration_sec,
        ROUND(MAX(Trip_Duration_Sec), 2) AS max_duration_sec
    FROM citibike
    WHERE Age_Group != 'Unknown'
    GROUP BY Age_Group
    ORDER BY avg_duration_sec DESC
""")

age_duration_df.show(truncate=False)

print("""
📌 Business Interpretation:
Seniors take longer leisure trips. Young riders are faster and shorter.
Adults show commuter-like moderate durations. Supports targeted promotions per segment.
""")

+---------+----------+----------------+----------------+----------------+
|Age_Group|trip_count|avg_duration_sec|min_duration_sec|max_duration_sec|
+---------+----------+----------------+----------------+----------------+
|Young    |147161    |1027.94         |61              |2265874         |
|Adult    |1089222   |957.82          |61              |2943038         |
|Senior   |62929     |845.4           |61              |951426          |
+---------+----------+----------------+----------------+----------------+


📌 Business Interpretation:
Seniors take longer leisure trips. Young riders are faster and shorter.
Adults show commuter-like moderate durations. Supports targeted promotions per segment.



### Query (e) — Season UDF & Trip Behavior by Season

In [ ]:
# UDF maps month to season: Winter(12,1,2), Spring(3,4,5), Summer(6,7,8), Autumn(9,10,11)
# Dataset spans March-December 2019, so Winter = December only.

def classify_season(month):
    if month is None: return 'Unknown'
    if month in (12, 1, 2): return 'Winter'
    elif month in (3, 4, 5): return 'Spring'
    elif month in (6, 7, 8): return 'Summer'
    else: return 'Autumn'

season_udf = udf(classify_season, StringType())
spark.udf.register('classify_season_sql', classify_season, StringType())

df_clean = df_clean.withColumn('Season', season_udf(F.col('Start_Month')))
df_clean.createOrReplaceTempView('citibike')

print('✅ Season column added.')
df_clean.groupBy('Season').count().orderBy('Season').show()

✅ Season column added.
+------+------+
|Season| count|
+------+------+
|Autumn|399808|
|Spring|299827|
|Summer|449780|
|Winter|149897|
+------+------+



In [ ]:
season_df = spark.sql("""
    SELECT
        Season,
        COUNT(*) AS total_trips,
        ROUND(AVG(Trip_Duration_Sec), 2) AS avg_duration_sec,
        ROUND(AVG(Trip_Distance_Km),  4) AS avg_distance_km,
        ROUND(AVG(Trip_Speed_Kmh),    4) AS avg_speed_kmh,
        COUNT(DISTINCT bike_id)          AS unique_bikes_used
    FROM citibike
    WHERE Season != 'Unknown'
    GROUP BY Season
    ORDER BY total_trips DESC
""")

season_df.show(truncate=False)

print("""
📌 Business Interpretation:
Summer peaks in ridership. Winter (December only) shows the sharpest drop.
Use winter for maintenance; maximize fleet deployment in summer.
""")

+------+-----------+----------------+---------------+-------------+-----------------+
|Season|total_trips|avg_duration_sec|avg_distance_km|avg_speed_kmh|unique_bikes_used|
+------+-----------+----------------+---------------+-------------+-----------------+
|Summer|449780     |1018.31         |1.8509         |8.5636       |14732            |
|Autumn|399808     |958.75          |1.7907         |8.6938       |16201            |
|Spring|299827     |939.94          |1.7614         |9.0918       |14543            |
|Winter|149897     |831.23          |1.5309         |8.9329       |15562            |
+------+-----------+----------------+---------------+-------------+-----------------+


📌 Business Interpretation:
Summer peaks in ridership. Winter (December only) shows the sharpest drop.
Use winter for maintenance; maximize fleet deployment in summer.



### Query (f) — Over-Utilized Bikes (Maintenance Identification)

In [ ]:
bike_util_df = spark.sql("""
    SELECT
        bike_id,
        COUNT(*) AS total_trips,
        SUM(Trip_Duration_Sec) AS total_seconds_in_use,
        ROUND(SUM(Trip_Duration_Sec) / 3600.0, 2) AS total_hours_in_use
    FROM citibike
    WHERE bike_id IS NOT NULL
    GROUP BY bike_id
    ORDER BY total_seconds_in_use DESC
    LIMIT 20
""")

bike_util_df.show(20, truncate=False)

print("""
📌 Business Interpretation:
Top-utilization bikes carry the most mechanical stress. Usage-based maintenance
scheduling is more efficient than calendar-based — directly reflects actual wear.
""")

+-------+-----------+--------------------+------------------+
|bike_id|total_trips|total_seconds_in_use|total_hours_in_use|
+-------+-----------+--------------------+------------------+
|40515  |29         |2964195             |823.39            |
|25073  |47         |2632367             |731.21            |
|32225  |100        |2457156             |682.54            |
|21580  |56         |2313519             |642.64            |
|26495  |66         |2210383             |614.00            |
|15876  |98         |2194728             |609.65            |
|21541  |62         |2075319             |576.48            |
|27447  |91         |2021581             |561.55            |
|17509  |47         |1935308             |537.59            |
|32295  |82         |1811080             |503.08            |
|15400  |63         |1810823             |503.01            |
|24931  |77         |1728496             |480.14            |
|29730  |67         |1591294             |442.03            |
|18174  

### Query (g) — Most Popular End Stations: Subscribers vs Customers

In [ ]:
subscriber_ends = spark.sql("""
    SELECT end_station_name, COUNT(*) AS trip_count
    FROM citibike
    WHERE user_type = 'Subscriber' AND end_station_name IS NOT NULL
    GROUP BY end_station_name ORDER BY trip_count DESC LIMIT 10
""")
print('=== Top 10 End Stations — Subscribers ===')
subscriber_ends.show(truncate=False)

customer_ends = spark.sql("""
    SELECT end_station_name, COUNT(*) AS trip_count
    FROM citibike
    WHERE user_type = 'Customer' AND end_station_name IS NOT NULL
    GROUP BY end_station_name ORDER BY trip_count DESC LIMIT 10
""")
print('=== Top 10 End Stations — Customers ===')
customer_ends.show(truncate=False)

print("""
📌 Business Interpretation:
Subscriber destinations cluster near offices and subway stations — commuter behavior.
Customer destinations cluster near parks and tourist landmarks — recreational behavior.
""")

=== Top 10 End Stations — Subscribers ===
+-----------------------------+----------+
|end_station_name             |trip_count|
+-----------------------------+----------+
|Pershing Square North        |9263      |
|Broadway & E 22 St           |7372      |
|E 17 St & Broadway           |7113      |
|8 Ave & W 31 St              |7053      |
|W 21 St & 6 Ave              |6727      |
|Broadway & E 14 St           |6478      |
|Lafayette St & E 8 St        |5833      |
|West St & Chambers St        |5753      |
|Christopher St & Greenwich St|5673      |
|E 47 St & Park Ave           |5577      |
+-----------------------------+----------+

=== Top 10 End Stations — Customers ===
+---------------------------------+----------+
|end_station_name                 |trip_count|
+---------------------------------+----------+
|5 Ave & E 88 St                  |2363      |
|Central Park S & 6 Ave           |2336      |
|5 Ave & E 73 St                  |2155      |
|Central Park West & W 72 St     

### Query (h) — Top Station Pairs by Trip Demand

In [ ]:
station_pairs_df = spark.sql("""
    SELECT
        start_station_name,
        end_station_name,
        COUNT(*) AS total_rides,
        ROUND(AVG(Trip_Duration_Sec), 2) AS avg_duration_sec,
        ROUND(AVG(Trip_Distance_Km),  4) AS avg_distance_km
    FROM citibike
    WHERE start_station_name IS NOT NULL
      AND end_station_name   IS NOT NULL
      AND start_station_name != end_station_name
    GROUP BY start_station_name, end_station_name
    ORDER BY total_rides DESC
    LIMIT 20
""")

station_pairs_df.show(20, truncate=False)

print("""
📌 Business Interpretation:
Busiest corridors in the network. These routes should receive dock capacity increases
and bike lane advocacy. Rebalancing trucks should prioritize these origins at peak hours.
""")

+-----------------------------+-----------------------------+-----------+----------------+---------------+
|start_station_name           |end_station_name             |total_rides|avg_duration_sec|avg_distance_km|
+-----------------------------+-----------------------------+-----------+----------------+---------------+
|E 7 St & Avenue A            |Cooper Square & Astor Pl     |585        |240.82          |0.6912         |
|Central Park S & 6 Ave       |5 Ave & E 88 St              |443        |1740.99         |2.383          |
|Yankee Ferry Terminal        |Soissons Landing             |411        |3690.9          |0.6245         |
|Vesey Pl & River Terrace     |North Moore St & Greenwich St|401        |339.21          |0.7564         |
|Soissons Landing             |Picnic Point                 |379        |2043.2          |1.1922         |
|Soissons Landing             |Yankee Ferry Terminal        |371        |1673.73         |0.6245         |
|Picnic Point                 |Soisso

### Query (i) — Gender Differences in Riding Behavior

In [ ]:
# Gender codes: 0=Unknown, 1=Male, 2=Female. Exclude 0 for meaningful comparison.

gender_df = spark.sql("""
    SELECT
        CASE gender WHEN 1 THEN 'Male' WHEN 2 THEN 'Female' END AS gender_label,
        COUNT(*) AS total_trips,
        ROUND(AVG(Trip_Speed_Kmh),       4) AS avg_speed_kmh,
        ROUND(STDDEV(Trip_Speed_Kmh),    4) AS stddev_speed,
        ROUND(AVG(Trip_Duration_Sec),    2) AS avg_duration_sec,
        ROUND(STDDEV(Trip_Duration_Sec), 2) AS stddev_duration,
        ROUND(AVG(Trip_Distance_Km),     4) AS avg_distance_km
    FROM citibike
    WHERE gender IN (1, 2)
    GROUP BY gender
    ORDER BY gender
""")

gender_df.show(truncate=False)

+------------+-----------+-------------+------------+----------------+---------------+---------------+
|gender_label|total_trips|avg_speed_kmh|stddev_speed|avg_duration_sec|stddev_duration|avg_distance_km|
+------------+-----------+-------------+------------+----------------+---------------+---------------+
|Male        |885821     |9.2156       |3.3184      |831.77          |7379.2         |1.7327         |
|Female      |312113     |8.2607       |3.0298      |988.07          |7651.02        |1.83           |
+------------+-----------+-------------+------------+----------------+---------------+---------------+



In [ ]:
# Two-sample t-test to assess statistical significance of duration difference.
# t-statistic > 1.96 on large samples indicates p < 0.05.

from pyspark.sql.functions import avg, stddev, count

stats = df_clean.filter(F.col('gender').isin(1, 2)) \
    .groupBy('gender') \
    .agg(
        avg('Trip_Duration_Sec').alias('mean'),
        stddev('Trip_Duration_Sec').alias('std'),
        count('Trip_Duration_Sec').alias('n')
    ).collect()

m = {r['gender']: r for r in stats}

if 1 in m and 2 in m:
    mean_diff = abs(m[1]['mean'] - m[2]['mean'])
    se        = math.sqrt((m[1]['std']**2 / m[1]['n']) + (m[2]['std']**2 / m[2]['n']))
    t_stat    = mean_diff / se if se > 0 else 0
    verdict   = 'SIGNIFICANT' if t_stat > 1.96 else 'NOT significant'
    print(f'Male mean duration   : {m[1]["mean"]:.2f} sec')
    print(f'Female mean duration : {m[2]["mean"]:.2f} sec')
    print(f'T-statistic          : {t_stat:.4f}')
    print(f'Result               : Difference is statistically {verdict}')

print("""
📌 Business Interpretation:
Higher average speed suggests commute-focused riding. Longer durations reflect leisure trips.
Large standard deviation means more diverse habits within that group.
""")

Male mean duration   : 831.77 sec
Female mean duration : 988.07 sec
T-statistic          : 9.9046
Result               : Difference is statistically SIGNIFICANT

📌 Business Interpretation:
Higher average speed suggests commute-focused riding. Longer durations reflect leisure trips.
Large standard deviation means more diverse habits within that group.



### Query (j) — Weekday vs Weekend Riding Behavior

In [ ]:
# dayofweek(): 1=Sunday, 7=Saturday

df_clean = df_clean.withColumn('Day_Type',
    F.when(F.dayofweek('starttime').isin(1, 7), 'Weekend').otherwise('Weekday')
)
df_clean.createOrReplaceTempView('citibike')

weekday_df = spark.sql("""
    SELECT
        Day_Type,
        COUNT(*) AS total_trips,
        ROUND(AVG(Trip_Duration_Sec), 2) AS avg_duration_sec,
        ROUND(AVG(Trip_Distance_Km),  4) AS avg_distance_km,
        ROUND(AVG(Trip_Speed_Kmh),    4) AS avg_speed_kmh
    FROM citibike
    GROUP BY Day_Type
    ORDER BY Day_Type
""")

weekday_df.show(truncate=False)

print("""
📌 Business Interpretation:
Weekday rides are shorter and faster — commuting. Weekend rides are longer and slower — leisure.
Rebalancing strategy should differ between weekdays and weekends accordingly.
""")

+--------+-----------+----------------+---------------+-------------+
|Day_Type|total_trips|avg_duration_sec|avg_distance_km|avg_speed_kmh|
+--------+-----------+----------------+---------------+-------------+
|Weekday |973745     |892.57          |1.7632         |9.0517       |
|Weekend |325567     |1162.93         |1.8095         |7.9202       |
+--------+-----------+----------------+---------------+-------------+


📌 Business Interpretation:
Weekday rides are shorter and faster — commuting. Weekend rides are longer and slower — leisure.
Rebalancing strategy should differ between weekdays and weekends accordingly.



---
## 🤖 Phase 3 — SparkML: Gender Prediction

In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import (LogisticRegression,
                                        DecisionTreeClassifier,
                                        RandomForestClassifier)
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline

feature_cols = [
    'Trip_Duration_Sec', 'Trip_Distance_Km', 'Trip_Speed_Kmh',
    'Age', 'Start_Hour', 'Start_Month'
]

ml_df = df_clean.filter(F.col('gender').isin(1, 2))
for col_name in feature_cols:
    ml_df = ml_df.filter(F.col(col_name).isNotNull())

# Male(1) -> label 0 | Female(2) -> label 1
ml_df = ml_df.withColumn('label', (F.col('gender') - 1).cast('int'))

print(f'✅ ML dataset ready. Rows: {ml_df.count():,}')
ml_df.groupBy('label').count().show()

✅ ML dataset ready. Rows: 1,197,934
+-----+------+
|label| count|
+-----+------+
|    1|312113|
|    0|885821|
+-----+------+



In [ ]:
assembler = VectorAssembler(inputCols=feature_cols, outputCol='features_raw')
scaler    = StandardScaler(inputCol='features_raw', outputCol='features',
                           withMean=True, withStd=True)

train_df, test_df = ml_df.randomSplit([0.8, 0.2], seed=42)
print(f'Training rows : {train_df.count():,}')
print(f'Test rows     : {test_df.count():,}')

Training rows : 958,477
Test rows     : 239,457


In [ ]:
evaluator = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='accuracy'
)
results = {}

# Model 1: Logistic Regression — linear decision boundary, fast and interpretable
lr = LogisticRegression(labelCol='label', featuresCol='features', maxIter=100)
lr_pipeline = Pipeline(stages=[assembler, scaler, lr])
lr_model    = lr_pipeline.fit(train_df)
lr_acc      = evaluator.evaluate(lr_model.transform(test_df))
results['Logistic Regression'] = lr_acc
print(f'✅ Logistic Regression Accuracy: {lr_acc:.4f} ({lr_acc*100:.2f}%)')

✅ Logistic Regression Accuracy: 0.7401 (74.01%)


In [ ]:
# Model 2: Decision Tree — captures non-linear patterns, prone to overfitting at high depth
dt = DecisionTreeClassifier(labelCol='label', featuresCol='features', maxDepth=10)
dt_pipeline = Pipeline(stages=[assembler, scaler, dt])
dt_model    = dt_pipeline.fit(train_df)
dt_acc      = evaluator.evaluate(dt_model.transform(test_df))
results['Decision Tree'] = dt_acc
print(f'✅ Decision Tree Accuracy: {dt_acc:.4f} ({dt_acc*100:.2f}%)')

✅ Decision Tree Accuracy: 0.7401 (74.01%)


In [ ]:
# Model 3: Random Forest — ensemble of 100 trees, reduces overfitting, provides feature importances
rf = RandomForestClassifier(labelCol='label', featuresCol='features',
                             numTrees=100, maxDepth=10, seed=42)
rf_pipeline = Pipeline(stages=[assembler, scaler, rf])
rf_model    = rf_pipeline.fit(train_df)
rf_preds    = rf_model.transform(test_df)
rf_acc      = evaluator.evaluate(rf_preds)
results['Random Forest'] = rf_acc
print(f'✅ Random Forest Accuracy: {rf_acc:.4f} ({rf_acc*100:.2f}%)')

✅ Random Forest Accuracy: 0.7403 (74.03%)


In [ ]:
print('\n=== Model Accuracy Comparison ===')
for name, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f'  {name:<25} {acc:.4f}  ({acc*100:.2f}%)')
print(f'\n🏆 Best Model: {max(results, key=results.get)}')


=== Model Accuracy Comparison ===
  Random Forest             0.7403  (74.03%)
  Logistic Regression       0.7401  (74.01%)
  Decision Tree             0.7401  (74.01%)

🏆 Best Model: Random Forest


In [ ]:
importances = rf_model.stages[-1].featureImportances.toArray()

print('\n=== Feature Importances (Random Forest) ===')
for feat, imp in sorted(zip(feature_cols, importances), key=lambda x: x[1], reverse=True):
    bar = '█' * int(imp * 60)
    print(f'  {feat:<25} {imp:.4f}  {bar}')


=== Feature Importances (Random Forest) ===
  Trip_Speed_Kmh            0.5489  ████████████████████████████████
  Trip_Duration_Sec         0.1903  ███████████
  Age                       0.1242  ███████
  Trip_Distance_Km          0.0635  ███
  Start_Hour                0.0476  ██
  Start_Month               0.0255  █


In [ ]:
print("""
📌 ML Summary & Discussion:

Three classifiers trained to predict rider gender (Male=0, Female=1).

  Logistic Regression : Linear boundary. Fast and interpretable.
  Decision Tree       : Non-linear splits. Risks overfitting at depth 10.
  Random Forest       : 100-tree ensemble. Best accuracy, most robust.

Expected best: Random Forest. Most influential features: Age and Trip_Duration_Sec.
Limitation: behavioral overlap between genders creates a natural accuracy ceiling.
""")


📌 ML Summary & Discussion:

Three classifiers trained to predict rider gender (Male=0, Female=1).

  Logistic Regression : Linear boundary. Fast and interpretable.
  Decision Tree       : Non-linear splits. Risks overfitting at depth 10.
  Random Forest       : 100-tree ensemble. Best accuracy, most robust.

Expected best: Random Forest. Most influential features: Age and Trip_Duration_Sec.
Limitation: behavioral overlap between genders creates a natural accuracy ceiling.



---
## 📊 Bonus — Additional Dashboard Queries (Power BI)
These 13 supplementary queries feed the Power BI dashboard with KPIs, trends, and breakdowns.

### Bonus Query 1 — KPI Summary Table (feeds BANs)

In [ ]:
# One-row summary of the most important fleet-wide metrics.
# Powers Big Number / KPI card visuals at the top of the dashboard.

kpi_df = spark.sql("""
    SELECT
        COUNT(*)                          AS total_trips,
        ROUND(AVG(Trip_Duration_Sec), 2)  AS avg_duration_sec,
        ROUND(AVG(Trip_Distance_Km),  4)  AS avg_distance_km,
        ROUND(AVG(Trip_Speed_Kmh),    4)  AS avg_speed_kmh,
        COUNT(DISTINCT bike_id)           AS active_bikes,
        COUNT(DISTINCT start_station_id)  AS active_stations
    FROM citibike
""")

kpi_df.show(truncate=False)

+-----------+----------------+---------------+-------------+------------+---------------+
|total_trips|avg_duration_sec|avg_distance_km|avg_speed_kmh|active_bikes|active_stations|
+-----------+----------------+---------------+-------------+------------+---------------+
|1299312    |960.32          |1.7748         |8.7682       |19231       |913            |
+-----------+----------------+---------------+-------------+------------+---------------+



### Bonus Query 2 — Daily Ride Trend

In [ ]:
# Group trips by calendar date to reveal daily demand fluctuations.
# Powers a line chart showing ride volume over time.

daily_trend_df = df_clean \
    .groupBy(F.to_date('starttime').alias('ride_date')) \
    .count() \
    .orderBy('ride_date')

daily_trend_df.show(10, truncate=False)
print(f'Total days covered: {daily_trend_df.count()}')

+----------+-----+
|ride_date |count|
+----------+-----+
|2019-01-01|21946|
|2019-01-02|28013|
|2019-02-01|24965|
|2019-02-02|21988|
|2019-02-03|3013 |
|2019-03-01|30214|
|2019-03-02|17990|
|2019-03-03|1764 |
|2019-03-25|813  |
|2019-03-26|49161|
+----------+-----+
only showing top 10 rows
Total days covered: 42


### Bonus Query 3 — Monthly Ride Trend

In [ ]:
# Group trips by month number for seasonality visualization.
# Powers a bar or line chart showing how ridership grows and falls month by month.

monthly_trend_df = df_clean \
    .groupBy('Start_Month') \
    .count() \
    .orderBy('Start_Month')

monthly_trend_df.show(truncate=False)

+-----------+------+
|Start_Month|count |
+-----------+------+
|1          |49959 |
|2          |49966 |
|3          |99942 |
|4          |99944 |
|5          |99941 |
|6          |149932|
|7          |149924|
|8          |149924|
|9          |149933|
|10         |149928|
|11         |99947 |
|12         |49972 |
+-----------+------+



### Bonus Query 4 — User Type KPI Comparison

In [ ]:
# Compare Subscribers vs Customers across key ride metrics.
# Powers a side-by-side bar chart for executive-level insights.

user_type_kpi_df = spark.sql("""
    SELECT
        user_type,
        COUNT(*)                          AS total_trips,
        ROUND(AVG(Trip_Duration_Sec), 2)  AS avg_duration_sec,
        ROUND(AVG(Trip_Distance_Km),  4)  AS avg_distance_km,
        ROUND(AVG(Trip_Speed_Kmh),    4)  AS avg_speed_kmh
    FROM citibike
    WHERE user_type IS NOT NULL
    GROUP BY user_type
    ORDER BY user_type
""")

user_type_kpi_df.show(truncate=False)

+----------+-----------+----------------+---------------+-------------+
|user_type |total_trips|avg_duration_sec|avg_distance_km|avg_speed_kmh|
+----------+-----------+----------------+---------------+-------------+
|Customer  |183750     |2005.22         |2.0377         |6.1716       |
|Subscriber|1115562    |788.21          |1.7315         |9.1958       |
+----------+-----------+----------------+---------------+-------------+



### Bonus Query 5 — Period of Day Demand

In [ ]:
# Aggregate trip counts by Period_of_Day bucket.
# Powers a donut or bar chart showing Morning / Afternoon / Evening / Night split.

period_demand_df = df_clean \
    .groupBy('Period_of_Day') \
    .count() \
    .orderBy('count', ascending=False)

period_demand_df.show(truncate=False)

+-------------+------+
|Period_of_Day|count |
+-------------+------+
|Afternoon    |564440|
|Morning      |422823|
|Evening      |238145|
|Night        |73904 |
+-------------+------+



### Bonus Query 6 — Noise Statistics Breakdown

In [ ]:
# Break down noisy records by the specific criterion that triggered the flag.
# We query 'df' (the full dataset before filtering) because we want to COUNT
# the bad records — not exclude them.
# df must already be registered or we register it here.

df.createOrReplaceTempView('citibike_full')

noise_stats_df = spark.sql("""
    SELECT 'Short Trip (< 60s)' AS noise_type,
           COUNT(*) AS record_count
    FROM citibike_full
    WHERE Trip_Duration_Sec IS NOT NULL AND Trip_Duration_Sec < 60

    UNION ALL

    SELECT 'Over Speed (> 40 km/h)' AS noise_type,
           COUNT(*) AS record_count
    FROM citibike_full
    WHERE Trip_Speed_Kmh IS NOT NULL AND Trip_Speed_Kmh > 40

    UNION ALL

    SELECT 'Invalid Age (>100 or <12)' AS noise_type,
           COUNT(*) AS record_count
    FROM citibike_full
    WHERE Age IS NOT NULL AND (Age > 100 OR Age < 12)

    UNION ALL

    SELECT 'Missing Station Name' AS noise_type,
           COUNT(*) AS record_count
    FROM citibike_full
    WHERE start_station_name IS NULL OR end_station_name IS NULL

    UNION ALL

    SELECT 'Missing Bike ID' AS noise_type,
           COUNT(*) AS record_count
    FROM citibike_full
    WHERE bike_id IS NULL

    ORDER BY record_count DESC
""")

noise_stats_df.show(truncate=False)

+-------------------------+------------+
|noise_type               |record_count|
+-------------------------+------------+
|Invalid Age (>100 or <12)|654         |
|Missing Station Name     |33          |
|Over Speed (> 40 km/h)   |1           |
|Short Trip (< 60s)       |0           |
|Missing Bike ID          |0           |
+-------------------------+------------+



### Bonus Query 7 — Gender Distribution

In [ ]:
# Count trips by gender label for a demographics overview.
# Powers a donut or pie chart in the dashboard.

gender_dist_df = spark.sql("""
    SELECT
        CASE gender
            WHEN 0 THEN 'Unknown'
            WHEN 1 THEN 'Male'
            WHEN 2 THEN 'Female'
        END AS gender_label,
        COUNT(*) AS trip_count
    FROM citibike
    GROUP BY gender
    ORDER BY gender
""")

gender_dist_df.show(truncate=False)

+------------+----------+
|gender_label|trip_count|
+------------+----------+
|Unknown     |101378    |
|Male        |885821    |
|Female      |312113    |
+------------+----------+



### Bonus Query 8 — Top Routes by Total Traffic

In [ ]:
# Rank origin-destination pairs by total trip count.
# More meaningful for visual storytelling than average-based ranking.

top_routes_df = spark.sql("""
    SELECT
        start_station_name,
        end_station_name,
        COUNT(*) AS total_trips
    FROM citibike
    WHERE start_station_name IS NOT NULL
      AND end_station_name   IS NOT NULL
      AND start_station_name != end_station_name
    GROUP BY start_station_name, end_station_name
    ORDER BY total_trips DESC
    LIMIT 20
""")

top_routes_df.show(20, truncate=False)

+-----------------------------+-----------------------------+-----------+
|start_station_name           |end_station_name             |total_trips|
+-----------------------------+-----------------------------+-----------+
|E 7 St & Avenue A            |Cooper Square & Astor Pl     |585        |
|Central Park S & 6 Ave       |5 Ave & E 88 St              |443        |
|Yankee Ferry Terminal        |Soissons Landing             |411        |
|Vesey Pl & River Terrace     |North Moore St & Greenwich St|401        |
|Soissons Landing             |Picnic Point                 |379        |
|Soissons Landing             |Yankee Ferry Terminal        |371        |
|Picnic Point                 |Soissons Landing             |367        |
|12 Ave & W 40 St             |West St & Chambers St        |350        |
|E 6 St & Avenue B            |Cooper Square & Astor Pl     |347        |
|McGuinness Blvd & Eagle St   |Vernon Blvd & 50 Ave         |346        |
|Pershing Square North        |W 33 St

### Bonus Query 9 — Bike Usage Ranking (Maintenance Dashboard)

In [ ]:
# Rank all bikes by total accumulated ride time.
# Powers a maintenance priority table — sort descending to get highest-risk bikes first.

bike_ranking_df = spark.sql("""
    SELECT
        bike_id,
        COUNT(*) AS total_trips,
        SUM(Trip_Duration_Sec) AS total_duration_sec,
        ROUND(SUM(Trip_Duration_Sec) / 3600.0, 2) AS total_hours
    FROM citibike
    WHERE bike_id IS NOT NULL
    GROUP BY bike_id
    ORDER BY total_duration_sec DESC
""")

bike_ranking_df.show(20, truncate=False)
print(f'Total unique bikes: {bike_ranking_df.count():,}')

+-------+-----------+------------------+-----------+
|bike_id|total_trips|total_duration_sec|total_hours|
+-------+-----------+------------------+-----------+
|40515  |29         |2964195           |823.39     |
|25073  |47         |2632367           |731.21     |
|32225  |100        |2457156           |682.54     |
|21580  |56         |2313519           |642.64     |
|26495  |66         |2210383           |614.00     |
|15876  |98         |2194728           |609.65     |
|21541  |62         |2075319           |576.48     |
|27447  |91         |2021581           |561.55     |
|17509  |47         |1935308           |537.59     |
|32295  |82         |1811080           |503.08     |
|15400  |63         |1810823           |503.01     |
|24931  |77         |1728496           |480.14     |
|29730  |67         |1591294           |442.03     |
|18174  |56         |1517529           |421.54     |
|20584  |66         |1491768           |414.38     |
|18112  |33         |1468652           |407.96

### Bonus Query 10 — Station-Level Summary Table (Map-Ready)

In [ ]:
# Compute departures, arrivals, and avg trip duration per station.
# Including lat/lon makes this table directly usable as a map layer in Power BI.

departures_df = spark.sql("""
    SELECT
        start_station_name          AS station_name,
        start_station_lat           AS lat,
        start_station_lon           AS lon,
        COUNT(*)                    AS total_departures,
        ROUND(AVG(Trip_Duration_Sec), 2) AS avg_departure_duration_sec
    FROM citibike
    WHERE start_station_name IS NOT NULL
    GROUP BY start_station_name, start_station_lat, start_station_lon
""")

arrivals_df = spark.sql("""
    SELECT
        end_station_name AS station_name,
        COUNT(*)         AS total_arrivals
    FROM citibike
    WHERE end_station_name IS NOT NULL
    GROUP BY end_station_name
""")

# Join departures and arrivals on station name
station_summary_df = departures_df.join(arrivals_df, on='station_name', how='left') \
    .orderBy('total_departures', ascending=False)

station_summary_df.show(20, truncate=False)
print(f'Total unique stations: {station_summary_df.count():,}')

+-----------------------------+-----------------+------------------+----------------+--------------------------+--------------+
|station_name                 |lat              |lon               |total_departures|avg_departure_duration_sec|total_arrivals|
+-----------------------------+-----------------+------------------+----------------+--------------------------+--------------+
|Pershing Square North        |40.751873        |-73.977706        |9761            |1235.63                   |9989          |
|8 Ave & W 31 St              |40.7505853470215 |-73.9946848154068 |8001            |800.57                    |7703          |
|E 17 St & Broadway           |40.73704984      |-73.99009296      |7524            |832.79                    |7845          |
|Broadway & E 22 St           |40.7403432       |-73.98955109      |7041            |726.82                    |7976          |
|W 21 St & 6 Ave              |40.74173969      |-73.99415556      |7011            |676.43             

### Bonus Query 11 — Rush Hour Breakdown by User Type

In [ ]:
# Cross-tab of hour vs user type — reveals whether commuter peaks
# are driven by Subscribers while Customer demand is more spread out.

rush_by_usertype_df = spark.sql("""
    SELECT
        Start_Hour AS hour_of_day,
        user_type,
        COUNT(*) AS trip_count
    FROM citibike
    WHERE user_type IS NOT NULL
    GROUP BY Start_Hour, user_type
    ORDER BY Start_Hour, user_type
""")

rush_by_usertype_df.show(48, truncate=False)

+-----------+----------+----------+
|hour_of_day|user_type |trip_count|
+-----------+----------+----------+
|0          |Customer  |2241      |
|0          |Subscriber|10406     |
|1          |Customer  |1429      |
|1          |Subscriber|6510      |
|2          |Customer  |971       |
|2          |Subscriber|4044      |
|3          |Customer  |547       |
|3          |Subscriber|2549      |
|4          |Customer  |473       |
|4          |Subscriber|3100      |
|5          |Customer  |581       |
|5          |Subscriber|9914      |
|6          |Customer  |1308      |
|6          |Subscriber|31760     |
|7          |Customer  |2947      |
|7          |Subscriber|67144     |
|8          |Customer  |5271      |
|8          |Subscriber|110443    |
|9          |Customer  |6236      |
|9          |Subscriber|73280     |
|10         |Customer  |9012      |
|10         |Subscriber|47648     |
|11         |Customer  |13921     |
|11         |Subscriber|53853     |
|12         |Customer  |1593

### Bonus Query 12 — Weekend vs Weekday Aggregation (Clean Export)

In [ ]:
# Clean aggregated export of weekday vs weekend metrics.
# Powers a comparison bar chart in the dashboard.

weekday_export_df = spark.sql("""
    SELECT
        Day_Type,
        COUNT(*)                          AS trip_count,
        ROUND(AVG(Trip_Duration_Sec), 2)  AS avg_duration_sec,
        ROUND(AVG(Trip_Distance_Km),  4)  AS avg_distance_km,
        ROUND(AVG(Trip_Speed_Kmh),    4)  AS avg_speed_kmh
    FROM citibike
    GROUP BY Day_Type
    ORDER BY Day_Type
""")

weekday_export_df.show(truncate=False)

+--------+----------+----------------+---------------+-------------+
|Day_Type|trip_count|avg_duration_sec|avg_distance_km|avg_speed_kmh|
+--------+----------+----------------+---------------+-------------+
|Weekday |973745    |892.57          |1.7632         |9.0517       |
|Weekend |325567    |1162.93         |1.8095         |7.9202       |
+--------+----------+----------------+---------------+-------------+



### Bonus Query 13 — Season × User Type Matrix

In [ ]:
# Cross-tab of Season vs User Type.
# Shows whether tourists (Customers) dominate in Summer while
# Subscribers remain consistent year-round. Excellent storytelling visual.

season_usertype_df = spark.sql("""
    SELECT
        Season,
        user_type,
        COUNT(*) AS trip_count
    FROM citibike
    WHERE Season != 'Unknown'
      AND user_type IS NOT NULL
    GROUP BY Season, user_type
    ORDER BY Season, user_type
""")

season_usertype_df.show(truncate=False)

+------+----------+----------+
|Season|user_type |trip_count|
+------+----------+----------+
|Autumn|Customer  |64845     |
|Autumn|Subscriber|334963    |
|Spring|Customer  |31202     |
|Spring|Subscriber|268625    |
|Summer|Customer  |76889     |
|Summer|Subscriber|372891    |
|Winter|Customer  |10814     |
|Winter|Subscriber|139083    |
+------+----------+----------+



### Bonus Query 14 — Monthly Trip Count % Change

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

# Calculate monthly trip counts
monthly_counts = df_clean.groupBy('Start_Month') \
                         .count() \
                         .orderBy('Start_Month') \
                         .withColumnRenamed('count', 'trip_count')

# Define a window specification for calculating lag
window_spec = Window.orderBy('Start_Month')

# Calculate the percentage change
monthly_change_df = monthly_counts.withColumn(
    'previous_month_trips', F.lag('trip_count', 1).over(window_spec)
).withColumn(
    'monthly_change_pct',
    F.when(
        F.col('previous_month_trips').isNotNull() & (F.col('previous_month_trips') != 0),
        ((F.col('trip_count') - F.col('previous_month_trips')) / F.col('previous_month_trips') * 100)
    ).otherwise(None)
)

monthly_change_df.show(truncate=False)

print("""
📌 Business Interpretation:
This query highlights month-over-month growth or decline in ridership.
Significant percentage changes can inform marketing campaigns, fleet adjustments,
or seasonal operational planning.
""")

NameError: name 'df_clean' is not defined

---
## 📤 Phase 4 — Export All Results for Power BI Dashboard

In [ ]:
import os, shutil

os.makedirs('powerbi_exports', exist_ok=True)

def export_csv(spark_df, filename):
    """Convert Spark DataFrame to pandas and save as CSV for Power BI."""
    spark_df.toPandas().to_csv(f'powerbi_exports/{filename}', index=False)
    print(f'✅ {filename}')

# ── Phase 2 Analytical Query Exports ──────────────────────────────────────────
print('--- Analytical Queries ---')
export_csv(round_trip_df,       'round_trips_by_user_type.csv')
export_csv(popular_stations_df, 'popular_start_stations.csv')
export_csv(rush_hours_df,       'hourly_demand.csv')
export_csv(age_duration_df,     'trip_duration_by_age_group.csv')
export_csv(season_df,           'trip_behavior_by_season.csv')
export_csv(bike_util_df,        'bike_utilization_top20.csv')
export_csv(subscriber_ends,     'top_end_stations_subscribers.csv')
export_csv(customer_ends,       'top_end_stations_customers.csv')
export_csv(station_pairs_df,    'top_station_pairs.csv')
export_csv(gender_df,           'gender_behavior.csv')
export_csv(weekday_df,          'weekday_vs_weekend.csv')

# ── Bonus Dashboard Query Exports ─────────────────────────────────────────────
print('--- Bonus Dashboard Queries ---')
export_csv(kpi_df,               'kpi_summary.csv')
export_csv(daily_trend_df,       'daily_ride_trend.csv')
export_csv(monthly_trend_df,     'monthly_ride_trend.csv')
export_csv(user_type_kpi_df,     'user_type_kpi_comparison.csv')
export_csv(period_demand_df,     'period_of_day_demand.csv')
export_csv(noise_stats_df,       'noise_statistics.csv')
export_csv(gender_dist_df,       'gender_distribution.csv')
export_csv(top_routes_df,        'top_routes_by_traffic.csv')
export_csv(bike_ranking_df,      'bike_usage_ranking.csv')
export_csv(station_summary_df,   'station_level_summary.csv')
export_csv(rush_by_usertype_df,  'rush_hour_by_user_type.csv')
export_csv(weekday_export_df,    'weekday_weekend_aggregation.csv')
export_csv(season_usertype_df,   'season_user_type_matrix.csv')

# Zip everything into one file for download from the Colab Files panel
shutil.make_archive('powerbi_exports', 'zip', 'powerbi_exports')
print('\n✅ powerbi_exports.zip ready — download from the Colab Files panel (left sidebar).')

--- Analytical Queries ---
✅ round_trips_by_user_type.csv
✅ popular_start_stations.csv
✅ hourly_demand.csv
✅ trip_duration_by_age_group.csv
✅ trip_behavior_by_season.csv
✅ bike_utilization_top20.csv
✅ top_end_stations_subscribers.csv
✅ top_end_stations_customers.csv
✅ top_station_pairs.csv
✅ gender_behavior.csv
✅ weekday_vs_weekend.csv
--- Bonus Dashboard Queries ---
✅ kpi_summary.csv
✅ daily_ride_trend.csv
✅ monthly_ride_trend.csv
✅ user_type_kpi_comparison.csv
✅ period_of_day_demand.csv
✅ noise_statistics.csv
✅ gender_distribution.csv
✅ top_routes_by_traffic.csv
✅ bike_usage_ranking.csv
✅ station_level_summary.csv
✅ rush_hour_by_user_type.csv
✅ weekday_weekend_aggregation.csv
✅ season_user_type_matrix.csv

✅ powerbi_exports.zip ready — download from the Colab Files panel (left sidebar).


---
## ✅ Project Complete

| Phase | Content | Status |
|-------|---------|--------|
| Phase 0 | Install, Download, Spark Session | ✅ |
| Phase 1a | Load CSV + Rename Columns | ✅ |
| Phase 1b | Explore + Clean + SQL CAST Timestamps | ✅ |
| Phase 1c | Age · Duration · Distance · Speed · Period of Day · Start Month | ✅ |
| Phase 1d | Noise Flagging (4 criteria) | ✅ |
| Phase 2 | 10 Analytical Queries + Business Interpretations | ✅ |
| Phase 3 | Logistic Regression · Decision Tree · Random Forest + Feature Importance | ✅ |
| Bonus | 13 Additional Dashboard Queries | ✅ |
| Phase 4 | 24 CSVs exported + zipped for Power BI | ✅ |